# 02 - Data ETL and Data Processing

**Session 2 assignment — Minh Nguyen**

Goal (per the session deck): explore and clean the streaming dataset based on my own
understanding of what the downstream analysis needs, save the cleaned files for
later sessions, and commit the result to the repo.

Starting point: the in-session reference notebook walked through the standard
clean-up steps (missing values, duplicates, format standardization). On top of that,
this version adds a few things I found while exploring the data myself — most
notably a real formatting inconsistency in the `IMDb` column that the reference
version didn't handle, plus a bit more validation.

## 1. Load data

In [1]:
import sys
!{sys.executable} -m pip install pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 10.3 MB/s  0:00:00 10.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 9.8 MB/s  0:00:00m 10.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]━━━━ 1/2 [pandas]


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Folders the loader will check, in order. Edit this list (or just hardcode
# DATA_DIR = Path("/full/path/to/Data")) if auto-detect doesn't find your files.
CANDIDATE_DIRS = [
    Path("."),
    Path("data"),
    Path("Data"),
    Path("../data"),
    Path("../Data"),
]

REQUIRED_FILES = ["MoviesOnStreamingPlatforms.csv", "TVShowsOnStreamingPlatforms.csv"]

def find_data_dir(filenames):
    for d in CANDIDATE_DIRS:
        if all((d / f).exists() for f in filenames):
            return d
    raise FileNotFoundError(
        "Could not auto-locate the data folder in: "
        f"{[str(d) for d in CANDIDATE_DIRS]}. "
        "Set DATA_DIR manually to the folder containing the two CSV files."
    )

DATA_DIR = find_data_dir(REQUIRED_FILES)
print(f"Using data folder: {DATA_DIR.resolve()}")

def load_csv(filename: str) -> pd.DataFrame:
    fp = DATA_DIR / filename
    print(f"Loaded: {fp}")
    return pd.read_csv(fp)

df_movie = load_csv("MoviesOnStreamingPlatforms.csv")
df_tv = load_csv("TVShowsOnStreamingPlatforms.csv")

print("df_movie shape:", df_movie.shape)
print("df_tv shape:", df_tv.shape)
df_movie.head()


Using data folder: /Users/minh1802/Downloads/streaming-bi-template/Data
Loaded: MoviesOnStreamingPlatforms.csv
Loaded: TVShowsOnStreamingPlatforms.csv
df_movie shape: (9515, 15)
df_tv shape: (5368, 15)


,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type,Genre,Country,Language,IMDb,IMDb_ID
0,1,The Irishman,2019,18+,98/100,1,0,0,0,0,"Biography, Crime, Drama",United States,"English, Italian, Latin, Spanish, German",7.8,tt1302006
1,2,Dangal,2016,7+,97/100,1,0,0,0,0,"Action, Biography, Drama","India, United States","Hindi, English",8.3,tt5074352
2,3,David Attenborough: A Life on Our Planet,2020,7+,95/100,1,0,0,0,0,"Documentary, Biography",United Kingdom,English,8.9,tt11989890
3,4,Lagaan: Once Upon a Time in India,2001,7+,94/100,1,0,0,0,0,"Drama, Musical, Sport","India, United States","Hindi, English",8.1,tt0169102
4,5,Roma,2018,18+,94/100,1,0,0,0,0,Drama,"Mexico, United States","Spanish, Mixtec, English, Japanese, German, Fr...",7.6,tt6155172


## 2. First look

Before touching anything, I want to see what dtypes pandas inferred and spot-check
a few columns for inconsistent formatting — this is what "explore the dataset based
on your own understanding" means in practice: don't just run the standard cleaning
steps, actually look at the values first.

In [3]:
print("Movies dtypes:")
print(df_movie.dtypes)
print("\nTV dtypes:")
print(df_tv.dtypes)

print("\nMovies — Age unique values:", df_movie["Age"].dropna().unique()[:10] if "Age" in df_movie.columns else "n/a")
print("Movies — Type unique values:", df_movie["Type"].unique()[:10] if "Type" in df_movie.columns else "n/a")

# IMDb is the one I want to flag: it looks numeric but pandas left it as
# object dtype, which usually means there's a mix of formats hiding in there.
if "IMDb" in df_tv.columns:
    print("\nTV — IMDb dtype:", df_tv["IMDb"].dtype)
    print("TV — sample IMDb values:", df_tv["IMDb"].dropna().unique()[:15])


Movies dtypes:
ID                   int64
Title                  str
Year                 int64
Age                    str
Rotten Tomatoes        str
Netflix              int64
Hulu                 int64
Prime Video          int64
Disney+              int64
Type                 int64
Genre                  str
Country                str
Language               str
IMDb               float64
IMDb_ID                str
dtype: object

TV dtypes:
ID                 int64
Title                str
Year               int64
Age                  str
Rotten Tomatoes      str
Netflix            int64
Hulu               int64
Prime Video        int64
Disney+            int64
Type               int64
Genre                str
Country              str
Language             str
IMDb                 str
IMDb_ID              str
dtype: object

Movies — Age unique values: <StringArray>
['18+', '7+', '13+', '16+', 'all']
Length: 5, dtype: str
Movies — Type unique values: [0]

TV — IMDb dtype: str
TV — sampl

## 3. Handle missing data

In [4]:
# Quick missingness overview (share of missing values per column)
def missingness(df: pd.DataFrame) -> pd.Series:
    return df.isna().mean().sort_values(ascending=False)

print("Missingness (Movies):")
display(missingness(df_movie))

print("\nMissingness (TV Shows):")
display(missingness(df_tv))


Missingness (Movies):


Age                0.438991
Language           0.366684
Genre              0.366264
Country            0.328744
IMDb               0.292486
IMDb_ID            0.286705
Rotten Tomatoes    0.000736
ID                 0.000000
Title              0.000000
Year               0.000000
Netflix            0.000000
Hulu               0.000000
Prime Video        0.000000
Disney+            0.000000
Type               0.000000
dtype: float64


Missingness (TV Shows):


Language           0.619411
Genre              0.588301
IMDb_ID            0.499255
Country            0.497578
Age                0.396237
IMDb               0.162072
ID                 0.000000
Title              0.000000
Year               0.000000
Rotten Tomatoes    0.000000
Netflix            0.000000
Hulu               0.000000
Prime Video        0.000000
Disney+            0.000000
Type               0.000000
dtype: float64

In [5]:
# Parsing / conversion helpers

def add_rotten_tomatoes_score(df: pd.DataFrame, source_col: str = "Rotten Tomatoes") -> pd.DataFrame:
    """Convert '98/100' -> 98.0 (numeric). Leaves NaN if missing/unparseable."""
    if source_col in df.columns:
        df["RottenTomatoes_Score"] = pd.to_numeric(
            df[source_col].astype(str).str.extract(r"(\d+)")[0],
            errors="coerce"
        )
    return df

def add_age_min(df: pd.DataFrame, source_col: str = "Age") -> pd.DataFrame:
    """Convert '18+' -> 18, '7+' -> 7 (numeric). Leaves NaN if missing/unparseable."""
    if source_col in df.columns:
        df["Age_Min"] = pd.to_numeric(
            df[source_col].astype(str).str.extract(r"(\d+)")[0],
            errors="coerce"
        )
    return df

def clean_imdb_score(df: pd.DataFrame, source_col: str = "IMDb") -> pd.DataFrame:
    """
    Real inconsistency I found while exploring: most rows store IMDb as a plain
    float ('9.0', '7.8'), but some rows are strings like '9.0/10'. That mix is
    exactly why pandas keeps the whole column as object dtype instead of float —
    and it would silently break .mean()/.describe() or any numeric comparison.
    Strip the trailing '/10' (if present) before converting to numeric.
    """
    if source_col in df.columns:
        cleaned = (
            df[source_col].astype(str)
            .str.replace(r"\s*/\s*10\s*$", "", regex=True)
            .str.strip()
        )
        df["IMDb_Score"] = pd.to_numeric(cleaned, errors="coerce")
    return df

def standardize_title(df: pd.DataFrame, col: str = "Title") -> pd.DataFrame:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
    return df

def standardize_type(df: pd.DataFrame, col: str = "Type") -> pd.DataFrame:
    """The datasets encode Type as 0/1. Map to readable labels, keep the raw value."""
    if col in df.columns:
        df["Type_raw"] = df[col]
        df[col] = df[col].map({0: "movie", 1: "tv_show"}).fillna(df[col].astype(str))
    return df

def coerce_year(df: pd.DataFrame, col: str = "Year") -> pd.DataFrame:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def coerce_platform_flags(df: pd.DataFrame, platform_cols=None) -> pd.DataFrame:
    """Ensure platform columns are numeric 0/1 (safe for grouping & sums)."""
    if platform_cols is None:
        platform_cols = ["Netflix", "Hulu", "Prime Video", "Disney+"]
    for c in platform_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    return df

def clean_genre(df: pd.DataFrame, col: str = "Genre") -> pd.DataFrame:
    """
    Genre is a free-text, comma-separated field, which is exactly the kind of
    column the session deck flagged as prone to 'Drama' vs 'drama' vs 'Dramas'
    style inconsistency. Build a normalized version for grouping/filtering,
    while keeping the original column for display.
    """
    if col in df.columns:
        def _norm(value):
            if not isinstance(value, str):
                return value
            parts = [p.strip().title() for p in value.split(",") if p.strip()]
            return ", ".join(parts)
        df[f"{col}_clean"] = df[col].apply(_norm)
    return df

# Apply conversions to BOTH datasets
for _df in (df_movie, df_tv):
    add_rotten_tomatoes_score(_df)
    add_age_min(_df)
    clean_imdb_score(_df)
    standardize_title(_df)
    standardize_type(_df)
    coerce_year(_df)
    coerce_platform_flags(_df)
    clean_genre(_df)

df_movie[["Title", "Year", "Age", "Age_Min", "Rotten Tomatoes", "RottenTomatoes_Score",
          "IMDb", "IMDb_Score", "Type"]].head()


,Title,Year,Age,Age_Min,Rotten Tomatoes,RottenTomatoes_Score,IMDb,IMDb_Score,Type
0,The Irishman,2019,18+,18.0,98/100,98.0,7.8,7.8,movie
1,Dangal,2016,7+,7.0,97/100,97.0,8.3,8.3,movie
2,David Attenborough: A Life on Our Planet,2020,7+,7.0,95/100,95.0,8.9,8.9,movie
3,Lagaan: Once Upon a Time in India,2001,7+,7.0,94/100,94.0,8.1,8.1,movie
4,Roma,2018,18+,18.0,94/100,94.0,7.6,7.6,movie


In [6]:
# Decide which columns are numeric vs categorical AFTER conversions
def get_num_cat_cols(df: pd.DataFrame):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in df.columns if c not in num_cols]
    return num_cols, cat_cols

num_cols_movie, cat_cols_movie = get_num_cat_cols(df_movie)
num_cols_tv, cat_cols_tv = get_num_cat_cols(df_tv)

print("Movies numeric columns:", num_cols_movie)
print("Movies categorical columns:", cat_cols_movie)

print("\nTV numeric columns:", num_cols_tv)
print("TV categorical columns:", cat_cols_tv)


Movies numeric columns: ['ID', 'Year', 'Netflix', 'Hulu', 'Prime Video', 'Disney+', 'IMDb', 'RottenTomatoes_Score', 'Age_Min', 'IMDb_Score', 'Type_raw']
Movies categorical columns: ['Title', 'Age', 'Rotten Tomatoes', 'Type', 'Genre', 'Country', 'Language', 'IMDb_ID', 'Genre_clean']

TV numeric columns: ['ID', 'Year', 'Netflix', 'Hulu', 'Prime Video', 'Disney+', 'RottenTomatoes_Score', 'Age_Min', 'IMDb_Score', 'Type_raw']
TV categorical columns: ['Title', 'Age', 'Rotten Tomatoes', 'Type', 'Genre', 'Country', 'Language', 'IMDb', 'IMDb_ID', 'Genre_clean']


In [7]:
# Impute missing values (demo pattern)
# NOTE: In real BI pipelines, we usually impute only when it makes business sense.
# Here I impute supporting numeric/categorical columns, but I deliberately do NOT
# impute Title or Year — those are "must-have" columns (see next cell), so a
# missing value there should drop the row rather than get faked.

def impute_basic(df: pd.DataFrame) -> pd.DataFrame:
    num_cols, cat_cols = get_num_cat_cols(df)

    for c in num_cols:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    for c in cat_cols:
        if df[c].isna().any():
            mode = df[c].mode()
            fill_value = mode.iloc[0] if not mode.empty else "Unknown"
            df[c] = df[c].fillna(fill_value)

    return df

df_movie = impute_basic(df_movie)
df_tv = impute_basic(df_tv)
print("Imputation complete.")


Imputation complete.


In [8]:
# Drop rows missing must-have columns (chosen intentionally: Title and Year
# are required for almost every later analysis — genre trends, year-over-year
# catalog growth, etc. — so a row without them isn't usable, not just "messy".)
must_have = ["Title", "Year"]

df_movie_before = len(df_movie)
df_tv_before = len(df_tv)

df_movie = df_movie.dropna(subset=[c for c in must_have if c in df_movie.columns])
df_tv = df_tv.dropna(subset=[c for c in must_have if c in df_tv.columns])

print(f"Movies dropped (must-have): {df_movie_before - len(df_movie)}")
print(f"TV dropped (must-have): {df_tv_before - len(df_tv)}")


Movies dropped (must-have): 0
TV dropped (must-have): 0


In [9]:
# Check: remaining missing values (top 10 columns)
print("Remaining missing values (Movies):")
display(df_movie.isna().sum().sort_values(ascending=False).head(10))

print("\nRemaining missing values (TV):")
display(df_tv.isna().sum().sort_values(ascending=False).head(10))


Remaining missing values (Movies):


ID                      0
Title                   0
Type_raw                0
IMDb_Score              0
Age_Min                 0
RottenTomatoes_Score    0
IMDb_ID                 0
IMDb                    0
Language                0
Country                 0
dtype: int64


Remaining missing values (TV):


ID                      0
Title                   0
Type_raw                0
IMDb_Score              0
Age_Min                 0
RottenTomatoes_Score    0
IMDb_ID                 0
IMDb                    0
Language                0
Country                 0
dtype: int64

## 4. Remove duplicates

In [10]:
# Show duplicate records (entire-row duplicates)
dup_all_movies = df_movie[df_movie.duplicated(keep=False)]
dup_all_tv = df_tv[df_tv.duplicated(keep=False)]

print("Movies: # duplicated rows (entire row):", len(dup_all_movies))
print("TV: # duplicated rows (entire row):", len(dup_all_tv))

display(dup_all_movies.head(10))


Movies: # duplicated rows (entire row): 0
TV: # duplicated rows (entire row): 0


,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type,Genre,Country,Language,IMDb,IMDb_ID,RottenTomatoes_Score,Age_Min,IMDb_Score,Type_raw,Genre_clean


In [11]:
# Deduplicate by a stable key (recommended over whole-row dedup)
# Using (ID, Title) as the key — ID alone should already be unique per source
# file, but pairing it with Title catches any accidental ID collisions too.
def dedupe(df: pd.DataFrame, subset_cols):
    before = len(df)
    df2 = df.drop_duplicates(subset=[c for c in subset_cols if c in df.columns], keep="first")
    after = len(df2)
    print(f"Removed {before - after} duplicates using subset={subset_cols}")
    return df2

df_movie = dedupe(df_movie, ["ID", "Title"])
df_tv = dedupe(df_tv, ["ID", "Title"])


Removed 0 duplicates using subset=['ID', 'Title']
Removed 0 duplicates using subset=['ID', 'Title']


## 5. Standardize formats

In [12]:
# Build a standardized join key for Title, and trim Country/Language —
# these are free-text fields most likely to have stray whitespace from
# the original scrape (e.g. "United States " vs "United States").
for _df in (df_movie, df_tv):
    if "Title" in _df.columns:
        _df["Title_key"] = _df["Title"].astype(str).str.strip().str.lower()
    if "Country" in _df.columns:
        _df["Country"] = _df["Country"].astype(str).str.strip()
    if "Language" in _df.columns:
        _df["Language"] = _df["Language"].astype(str).str.strip()

df_movie[["Title", "Title_key", "Country", "Language"]].head()


,Title,Title_key,Country,Language
0,The Irishman,the irishman,United States,"English, Italian, Latin, Spanish, German"
1,Dangal,dangal,"India, United States","Hindi, English"
2,David Attenborough: A Life on Our Planet,david attenborough: a life on our planet,United Kingdom,English
3,Lagaan: Once Upon a Time in India,lagaan: once upon a time in india,"India, United States","Hindi, English"
4,Roma,roma,"Mexico, United States","Spanish, Mixtec, English, Japanese, German, Fr..."


In [13]:
# Preview cleaned Movies dataset
df_movie.head()


,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type,...,Country,Language,IMDb,IMDb_ID,RottenTomatoes_Score,Age_Min,IMDb_Score,Type_raw,Genre_clean,Title_key
0,1,The Irishman,2019,18+,98/100,1,0,0,0,movie,...,United States,"English, Italian, Latin, Spanish, German",7.8,tt1302006,98.0,18.0,7.8,0,"Biography, Crime, Drama",the irishman
1,2,Dangal,2016,7+,97/100,1,0,0,0,movie,...,"India, United States","Hindi, English",8.3,tt5074352,97.0,7.0,8.3,0,"Action, Biography, Drama",dangal
2,3,David Attenborough: A Life on Our Planet,2020,7+,95/100,1,0,0,0,movie,...,United Kingdom,English,8.9,tt11989890,95.0,7.0,8.9,0,"Documentary, Biography",david attenborough: a life on our planet
3,4,Lagaan: Once Upon a Time in India,2001,7+,94/100,1,0,0,0,movie,...,"India, United States","Hindi, English",8.1,tt0169102,94.0,7.0,8.1,0,"Drama, Musical, Sport",lagaan: once upon a time in india
4,5,Roma,2018,18+,94/100,1,0,0,0,movie,...,"Mexico, United States","Spanish, Mixtec, English, Japanese, German, Fr...",7.6,tt6155172,94.0,18.0,7.6,0,Drama,roma


In [14]:
# Preview cleaned TV dataset
df_tv.head()


,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type,...,Country,Language,IMDb,IMDb_ID,RottenTomatoes_Score,Age_Min,IMDb_Score,Type_raw,Genre_clean,Title_key
0,1,Breaking Bad,2008,18+,100/100,1,0,0,0,tv_show,...,United States,"American English, Spanish language in the Amer...",10.0,tt0903747,100,18.0,10.0,1,"Crime Television Series, Drama Television Seri...",breaking bad
1,2,Stranger Things,2016,16+,96/100,1,0,0,0,tv_show,...,United States,English,9.0,tt4574334,96,16.0,9.0,1,"Science Fiction Television Program, Horror Tel...",stranger things
2,3,Attack on Titan,2013,18+,95/100,1,1,0,0,tv_show,...,United States,English,9.0/10,tt0092400,95,18.0,9.0,1,Reality Television,attack on titan
3,4,Better Call Saul,2015,18+,94/100,1,0,0,0,tv_show,...,United States,English,10.0,tt3032476,94,18.0,10.0,1,"Legal Drama, Crime Television Series",better call saul
4,5,Dark,2017,16+,93/100,1,0,0,0,tv_show,...,Germany,German,10.0,tt5753856,93,16.0,10.0,1,"Science Fiction Television Program, Drama Tele...",dark


## 6. Validate ranges & business rules

In [15]:
def range_check(df: pd.DataFrame, col: str, lo, hi, name: str):
    if col in df.columns:
        bad = ~df[col].between(lo, hi) & df[col].notna()
        print(f"{name}: {col} out of [{lo}, {hi}]: {int(bad.sum())} rows")

print("--- Movies ---")
range_check(df_movie, "RottenTomatoes_Score", 0, 100, "Movies")
range_check(df_movie, "IMDb_Score", 0, 10, "Movies")
range_check(df_movie, "Year", 1900, 2026, "Movies")

print("\n--- TV Shows ---")
range_check(df_tv, "RottenTomatoes_Score", 0, 100, "TV")
range_check(df_tv, "IMDb_Score", 0, 10, "TV")
range_check(df_tv, "Year", 1900, 2026, "TV")


--- Movies ---
Movies: RottenTomatoes_Score out of [0, 100]: 0 rows
Movies: IMDb_Score out of [0, 10]: 0 rows
Movies: Year out of [1900, 2026]: 0 rows

--- TV Shows ---
TV: RottenTomatoes_Score out of [0, 100]: 0 rows
TV: IMDb_Score out of [0, 10]: 0 rows
TV: Year out of [1900, 2026]: 0 rows


## 7. Lightweight data quality report

One reusable summary cell — useful in every future session, not just this one.

In [16]:
def dq_report(frame: pd.DataFrame) -> pd.DataFrame:
    report = pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "n_missing": frame.isna().sum(),
        "pct_missing": frame.isna().mean().round(4),
        "n_unique": frame.nunique(dropna=False)
    })
    return report.sort_values(by=["pct_missing", "n_unique"], ascending=[False, True])

print("Data quality report — Movies")
display(dq_report(df_movie).head(20))

print("\nData quality report — TV Shows")
display(dq_report(df_tv).head(20))


Data quality report — Movies


,dtype,n_missing,pct_missing,n_unique
Type,str,0,0.0,1
Type_raw,int64,0,0.0,1
Netflix,int64,0,0.0,2
Hulu,int64,0,0.0,2
Prime Video,int64,0,0.0,2
Disney+,int64,0,0.0,2
Age_Min,float64,0,0.0,4
Age,str,0,0.0,5
IMDb,float64,0,0.0,69
IMDb_Score,float64,0,0.0,69



Data quality report — TV Shows


,dtype,n_missing,pct_missing,n_unique
Type,str,0,0.0,1
Type_raw,int64,0,0.0,1
Netflix,int64,0,0.0,2
Hulu,int64,0,0.0,2
Prime Video,int64,0,0.0,2
Disney+,int64,0,0.0,2
Age_Min,float64,0,0.0,4
Age,str,0,0.0,5
IMDb_Score,float64,0,0.0,73
Year,int64,0,0.0,78


## 8. Sanity checks and save cleaned data

In [17]:
def sanity_checks(df: pd.DataFrame, name: str):
    print(f"--- {name} ---")
    if "Year" in df.columns:
        print("Year min/max:", int(df["Year"].min()), int(df["Year"].max()))
    for c in ["Netflix", "Hulu", "Prime Video", "Disney+"]:
        if c in df.columns:
            print(f"{c} value counts:", df[c].value_counts().to_dict())
    if "RottenTomatoes_Score" in df.columns:
        print("RottenTomatoes_Score describe:")
        display(df["RottenTomatoes_Score"].describe())
    if "IMDb_Score" in df.columns:
        print("IMDb_Score describe:")
        display(df["IMDb_Score"].describe())

sanity_checks(df_movie, "Movies")
sanity_checks(df_tv, "TV Shows")


--- Movies ---
Year min/max: 1914 2021
Netflix value counts: {0: 5820, 1: 3695}
Hulu value counts: {0: 8468, 1: 1047}
Prime Video value counts: {0: 5402, 1: 4113}
Disney+ value counts: {0: 8593, 1: 922}
RottenTomatoes_Score describe:


count    9515.000000
mean       53.543878
std        13.192884
min        10.000000
25%        44.000000
50%        52.000000
75%        62.000000
max        98.000000
Name: RottenTomatoes_Score, dtype: float64

IMDb_Score describe:


count    9515.000000
mean        6.720494
std         1.815770
min         1.000000
25%         6.000000
50%         7.000000
75%         7.300000
max        10.000000
Name: IMDb_Score, dtype: float64

--- TV Shows ---
Year min/max: 1904 2021
Netflix value counts: {0: 3397, 1: 1971}
Hulu value counts: {0: 3747, 1: 1621}
Prime Video value counts: {0: 3537, 1: 1831}
Disney+ value counts: {0: 5017, 1: 351}
RottenTomatoes_Score describe:


count    5368.000000
mean       47.220380
std        19.555753
min        10.000000
25%        36.000000
50%        48.000000
75%        60.000000
max       100.000000
Name: RottenTomatoes_Score, dtype: float64

IMDb_Score describe:


count    5368.000000
mean        7.331967
std         1.728869
min         1.000000
25%         6.800000
50%         7.500000
75%         8.000000
max        10.000000
Name: IMDb_Score, dtype: float64

In [18]:
df_movie.to_csv(DATA_DIR / "MoviesOnStreamingPlatforms_Cleaned.csv", index=False)
df_tv.to_csv(DATA_DIR / "TVShowsOnStreamingPlatforms_Cleaned.csv", index=False)
print("Saved cleaned files to:", DATA_DIR.resolve())


Saved cleaned files to: /Users/minh1802/Downloads/streaming-bi-template/Data


## Notes — cleaning decisions (for the PR description / commit message)

- **Missing data:** imputed numeric columns with the median and categorical columns
  with the mode, *except* `Title` and `Year` — rows missing either of those were
  dropped instead, since both are required for the trend/genre analysis coming up
  in later sessions.
- **Duplicates:** deduplicated on `(ID, Title)` rather than whole-row matching, since
  a single mismatched field shouldn't let an otherwise-duplicate title through.
- **IMDb format bug (found during exploration, not in the original reference code):**
  the `IMDb` column mixes plain floats (`9.0`) with strings like `9.0/10`, which kept
  the whole column as text. Added `clean_imdb_score()` to strip the `/10` suffix and
  produce a proper numeric `IMDb_Score` column.
- **Genre normalization:** added a `Genre_clean` column (trimmed, title-cased,
  comma-separated) so genre filters/group-bys don't get split across
  `"drama"` / `"Drama"` / `"Dramas"` style variants, per the data-quality checklist
  in the session deck.
- **Validation:** checked `RottenTomatoes_Score` (0–100), `IMDb_Score` (0–10), and
  `Year` (1900–2026) for out-of-range values after cleaning.
- **Output:** `MoviesOnStreamingPlatforms_Cleaned.csv` and
  `TVShowsOnStreamingPlatforms_Cleaned.csv`, saved alongside the raw files in
  `DATA_DIR`, ready for the Session 3 EDA notebook.

**Before committing:** run every cell top to bottom in Jupyter so the saved outputs
reflect your actual data, then follow the branch → commit → PR steps from the
session deck (e.g. `git checkout -b feature/session2-data-cleaning`).